# Handle Class Imbalance

Consumes the leakage-safe, modeling-ready matrices saved by
`notebooks/feature-engineering.ipynb` (in `processed/`) and rebalances the
**training data only**.

**Why rebalance, and why only train?** Attrition is the minority class
(~16%). A model trained on the raw distribution can score ~84% accuracy by
always predicting "No" while catching zero leavers — useless for HR. We
oversample the minority in training so the model actually learns the leaver
signal. The **test set is left at its real-world ~16% rate** so evaluation
reflects production conditions (no leakage of synthetic rows into test).

**Two valid strategies, kept side by side for the modeling stage:**
- **SMOTE-resampled** `X_train_smote` / `y_train_smote` — for estimators with
  no native class weighting (e.g. Logistic Regression). *This stage produces
  these and persists them.*
- **Original** `X_train` / `y_train` (already in `processed/`) — used with
  `class_weight='balanced'` / `scale_pos_weight` on tree models.

Markdown cells are written as `# ` comment blocks so this `.py` can be pasted
straight into a notebook.

## 1. Setup & load

Load the four matrices emitted by the feature-engineering stage. Everything
here operates on the engineered + scaled features; no re-derivation needed.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
PROC = Path('processed')

X_train = pd.read_parquet(PROC / 'X_train.parquet')
X_test = pd.read_parquet(PROC / 'X_test.parquet')
y_train = pd.read_parquet(PROC / 'y_train.parquet')['Attrition']
y_test = pd.read_parquet(PROC / 'y_test.parquet')['Attrition']

print(f'X_train {X_train.shape} | attrition {y_train.mean():.3f}')
print(f'X_test  {X_test.shape} | attrition {y_test.mean():.3f}')

## 2. Confirm the imbalance

Quantify the skew before touching it. The imbalance ratio (~5.2:1) and the
"always-No" accuracy below are the reason accuracy is the wrong metric for
this problem — Recall (Yes) is the priority.

In [ ]:
train_counts = y_train.value_counts().sort_index()  # index 0 = No, 1 = Yes
imbalance_ratio = train_counts[0] / train_counts[1]
always_no_acc = train_counts[0] / len(y_train)

print(f'Class counts    No (stayed): {train_counts[0]}   Yes (left): {train_counts[1]}')
print(f'Imbalance ratio (No:Yes): {imbalance_ratio:.1f} : 1')
print(f'A classifier that always predicts "No" scores {always_no_acc:.1%} accuracy '
      f'while catching 0 leavers.')

# Before-SMOTE distribution
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No (Stayed)', 'Yes (Left)'], train_counts.values,
              color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
for bar, count in zip(bars, train_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
            f'{count}\n({count / len(y_train) * 100:.1f}%)',
            ha='center', fontweight='bold')
ax.set_title(f'Training Set — Before SMOTE\nImbalance ratio {imbalance_ratio:.1f}:1',
             fontweight='bold')
ax.set_ylabel('Number of Employees')
ax.set_ylim(0, max(train_counts.values) * 1.22)
plt.tight_layout()
plt.show()

## 3. Apply SMOTE (training data only)

SMOTE synthesises new minority-class examples by interpolating between each
minority point and its nearest minority neighbours, producing a 50/50 split.
Fit/resample on **train only**; the test set is never passed in. We rebuild a
DataFrame/Series afterwards so column names survive for SHAP in later stages.

In [ ]:
smote = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Restore labels (imblearn returns numpy / unnamed objects)
X_train_smote = pd.DataFrame(X_train_smote, columns=X_train.columns)
y_train_smote = pd.Series(y_train_smote, name='Attrition')

smote_counts = y_train_smote.value_counts().sort_index()
synthetic_added = smote_counts[1] - train_counts[1]

print(f'SMOTE applied (k_neighbors=5, random_state={RANDOM_STATE}).')
print(f'X_train_smote : {X_train_smote.shape}')
print(f'y_train_smote : {y_train_smote.shape}')
print(f'  No  (0): {smote_counts[0]}  ({smote_counts[0] / len(y_train_smote) * 100:.1f}%)')
print(f'  Yes (1): {smote_counts[1]}  ({smote_counts[1] / len(y_train_smote) * 100:.1f}%)')
print(f'Synthetic minority examples added: {synthetic_added}')
print(f'X_test unchanged: {X_test.shape}  (still {y_test.mean():.3f} attrition)')


## 4. Before / after comparison

Visualise the rebalance: the minority class is lifted to parity with the
majority while the majority count is unchanged.

In [ ]:
labels = ['No (Stayed)', 'Yes (Left)']
x = np.arange(len(labels))
width = 0.38

fig, ax = plt.subplots(figsize=(7, 4.5))
b1 = ax.bar(x - width / 2, train_counts.values, width,
            label='Before SMOTE', color='lightsteelblue', edgecolor='white')
b2 = ax.bar(x + width / 2, smote_counts.values, width,
            label='After SMOTE', color='mediumseagreen', edgecolor='white')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
                f'{int(bar.get_height())}', ha='center', fontweight='bold', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Number of Employees')
ax.set_title('Training Set Class Distribution — Before vs After SMOTE', fontweight='bold')
ax.set_ylim(0, max(smote_counts.values) * 1.18)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Persist outputs

The bridge to the modeling notebook. The original `X_train` / `y_train`
already live in `processed/` (saved by feature-engineering), so we only add
the SMOTE-resampled training set here. Downstream modeling chooses its input:
- **SMOTE** → `X_train_smote` / `y_train_smote` (e.g. Logistic Regression)
- **class-weight** → original `X_train` / `y_train` (Random Forest, XGBoost, LightGBM)

The test set (`X_test` / `y_test`) is never resaved — it stays exactly as the
feature-engineering stage left it.

In [ ]:
X_train_smote.to_parquet(PROC / 'X_train_smote.parquet')
y_train_smote.to_frame('Attrition').to_parquet(PROC / 'y_train_smote.parquet')

print('Saved:')
print(f'  {PROC / "X_train_smote.parquet"}  {X_train_smote.shape}')
print(f'  {PROC / "y_train_smote.parquet"}  {y_train_smote.shape}')

## 6. Verification

Asserts double as the stage's built-in test. They confirm the resample is
balanced, lossless in feature space, and that the test set was untouched.


In [ ]:
assert list(X_train_smote.columns) == list(X_train.columns), 'column mismatch after SMOTE'
assert X_train_smote.shape[1] == X_train.shape[1]

# Resampled training set is exactly balanced
assert smote_counts[0] == smote_counts[1], 'SMOTE did not produce a 50/50 split'

# Majority count unchanged; original distribution intact
assert smote_counts[0] == train_counts[0], 'majority class count changed unexpectedly'
assert (train_counts[0], train_counts[1]) == tuple(y_train.value_counts().sort_index())

# Test set untouched (no synthetic rows leaked in)
assert X_test.shape[0] == len(y_test)
assert abs(y_test.mean() - 0.16) < 0.02, 'test attrition rate drifted from real-world ~16%'

# No NaN / inf introduced by interpolation
assert not X_train_smote.isna().any().any(), 'X_train_smote has NaN'
assert np.isfinite(X_train_smote.to_numpy(dtype=float)).all(), 'X_train_smote has inf'

# Reload check — files are readable and shape-consistent
assert pd.read_parquet(PROC / 'X_train_smote.parquet').shape == X_train_smote.shape

print('All checks passed.')
print(f'  Before : {train_counts[0]} No / {train_counts[1]} Yes  ({imbalance_ratio:.1f}:1)')
print(f'  After  : {smote_counts[0]} No / {smote_counts[1]} Yes  (1.0:1)')
print(f'  Test set held out at {y_test.mean():.3f} attrition — evaluation stays realistic.')
